# Sistem Monitoring Aktivitas Tikus Sawah

**Alur kerja:**
1. Colab start → URL ngrok dikirim ke Apps Script → disimpan ke sel B2 Sheets
2. ESP32 #2 boot → baca URL via Apps Script (HTTP GET) → kirim data PIR tiap 10 menit
3. Colab proses fitur + Random Forest → klasifikasi + prediksi
4. Hasil → Google Sheets (logging) + MQTT (perintah ke ESP32 #1)

**Jalankan sel secara berurutan: SEL 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9**

## SEL 1 – Install library

In [ ]:
!pip install pyngrok flask paho-mqtt gspread google-auth joblib scikit-learn --quiet
print('Install selesai.')

Install selesai.


## SEL 2 – Konfigurasi (WAJIB diisi sebelum lanjut)

In [ ]:
# ================================================================
#  KONFIGURASI – sesuaikan semua nilai di bawah ini
# ================================================================

# --- Google Sheets ---
SHEET_NAME = 'Monitoring Tikus Prototipe Sawah'

# --- Google Apps Script ---
# URL permanen dari hasil deploy Apps Script
# Cara dapat: buka Sheets -> Extensions -> Apps Script -> Deploy -> Manage deployments -> salin URL
# URL bentuknya: https://script.google.com/macros/s/XXXX/exec
APPS_SCRIPT_URL = 'https://script.google.com/macros/s/AKfycbxnKxXivcs9P7tWvBVb3Ihma0lnh-KeRrB6VXMgoy08EaGjtYfjvJxg_T4wrU0ypnRQ/exec'  # <-- ganti ini (hanya sekali)

# --- MQTT ---
MQTT_BROKER    = 'broker.emqx.io'
MQTT_PORT      = 1883
MQTT_TOPIC     = 'sawah/cerdas/perintah'
MQTT_CLIENT_ID = 'colab_monitoring_tikus'

# --- Model Random Forest ---
# Upload file .pkl ke Colab: klik ikon folder sidebar kiri -> drag & drop
MODEL_PATH          = '/content/model_rf_klasifikasi.pkl'
MODEL_PREDIKSI_PATH = '/content/model_rf_prediksi.pkl'  # isi None jika tidak ada

# --- Frekuensi per klasifikasi (Hz) sesuai Tabel 7 ---
FREQ_MAP = {
    'rendah': {'freq_min': 20000, 'freq_max': 20167},
    'sedang': {'freq_min': 20167, 'freq_max': 20333},
    'tinggi': {'freq_min': 20333, 'freq_max': 20500},
}

# --- Ngrok ---
NGROK_AUTHTOKEN = '3DKoVT5MpUMLTmYEWK8jwtOizB7_7XuYTK4tVHWVyG43JN7Y5'  # <-- ganti ini (hanya sekali)

print('Konfigurasi disimpan.')

Konfigurasi disimpan.


## SEL 3 – Autentikasi Google

In [ ]:
from google.colab import auth
import gspread
from google.auth import default

auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)
print('Autentikasi Google berhasil.')

Autentikasi Google berhasil.


## SEL 4 – Setup Google Sheets

In [ ]:
try:
    spreadsheet = gc.open(SHEET_NAME)
    worksheet   = spreadsheet.sheet1
    print(f'Sheet "{SHEET_NAME}" sudah ada.')
except gspread.SpreadsheetNotFound:
    spreadsheet = gc.create(SHEET_NAME)
    worksheet   = spreadsheet.sheet1
    header = [
        'Tanggal',
        'Interval mulai',
        'Interval selesai',
        'Interval ke- (t)',
        'Jumlah Deteksi (count)',
        'Durasi',
        'Klasifikasi Aktivitas Tikus',
        'Frekuensi Ultrasonik',
        'Prediksi Aktivitas Tikus 1 Jam Ke depan'
    ]
    worksheet.append_row(header)
    print(f'Sheet "{SHEET_NAME}" berhasil dibuat.')

# Setup tab Info Server
try:
    info_sheet = spreadsheet.worksheet('Info Server')
    print('Tab "Info Server" sudah ada.')
except gspread.WorksheetNotFound:
    info_sheet = spreadsheet.add_worksheet(title='Info Server', rows=10, cols=3)
    info_sheet.append_row(['Keterangan', 'Nilai', 'Update Terakhir'])
    info_sheet.append_row(['URL Colab', 'belum tersedia', '-'])
    print('Tab "Info Server" berhasil dibuat.')

sheet_url = f'https://docs.google.com/spreadsheets/d/{spreadsheet.id}'
print(f'\nLink Google Sheets: {sheet_url}')
print(f'Spreadsheet ID    : {spreadsheet.id}')
print('\nSalin Spreadsheet ID di atas jika diperlukan.')

Sheet "Monitoring Tikus Prototipe Sawah" sudah ada.
Tab "Info Server" sudah ada.

Link Google Sheets: https://docs.google.com/spreadsheets/d/1O5snO0z4CbhGTdEZvjzE5QT5y4l_CuAQKSIS8qhM7Qo
Spreadsheet ID    : 1O5snO0z4CbhGTdEZvjzE5QT5y4l_CuAQKSIS8qhM7Qo

Salin Spreadsheet ID di atas jika diperlukan.


## SEL 5 – Load Model Random Forest

In [ ]:
import joblib
import os

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f'File "{MODEL_PATH}" tidak ditemukan.\n'
        'Upload file .pkl: klik ikon folder sidebar kiri Colab -> drag & drop.'
    )

model_klasifikasi = joblib.load(MODEL_PATH)
print(f'Model klasifikasi dimuat: {MODEL_PATH}')

model_prediksi = None
if MODEL_PREDIKSI_PATH and os.path.exists(MODEL_PREDIKSI_PATH):
    model_prediksi = joblib.load(MODEL_PREDIKSI_PATH)
    print(f'Model prediksi dimuat: {MODEL_PREDIKSI_PATH}')
else:
    print('Model prediksi tidak ada, kolom prediksi diisi "-".')

Model klasifikasi dimuat: /content/model_rf_klasifikasi.pkl
Model prediksi dimuat: /content/model_rf_prediksi.pkl


## SEL 6 – Setup MQTT

In [ ]:
import paho.mqtt.client as mqtt
import json

mqtt_client = mqtt.Client(client_id=MQTT_CLIENT_ID)

def on_connect(client, userdata, flags, rc):
    status = 'Terhubung' if rc == 0 else f'Gagal (rc={rc})'
    print(f'[MQTT] {status} ke {MQTT_BROKER}')

mqtt_client.on_connect = on_connect
mqtt_client.connect(MQTT_BROKER, MQTT_PORT, keepalive=60)
mqtt_client.loop_start()
print('[MQTT] Client aktif di background.')

/tmp/ipykernel_3571/1939592449.py:4: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  mqtt_client = mqtt.Client(client_id=MQTT_CLIENT_ID)


[MQTT] Client aktif di background.


## SEL 7 – Fungsi pemrosesan fitur & prediksi

In [ ]:
import numpy as np
import pandas as pd
from collections import deque
from datetime import datetime

riwayat_count  = deque(maxlen=6)
nomor_interval = 0


def hitung_fitur(count_sekarang, durasi):
    hist = list(riwayat_count)
    n    = len(hist)

    def lag(k):
        idx = n - k
        return hist[idx] if idx >= 0 else 0

    lag1 = lag(1); lag2 = lag(2); lag3 = lag(3)
    lag4 = lag(4); lag5 = lag(5); lag6 = lag(6)

    avg_30min = round(np.mean([lag1, lag2, lag3]), 4)
    avg_60min = round(np.mean([lag1, lag2, lag3, lag4, lag5, lag6]), 4)
    delta1    = count_sekarang - lag1

    fitur = {
    'count'      : count_sekarang,
    'durasi aktif': durasi,
    'lag1'       : lag1,
    'lag2'       : lag2,
    'lag3'       : lag3,
    'lag4'       : lag4,
    'lag5'       : lag5,
    'lag6'       : lag6,
    'avg30'      : avg_30min,
    'avg60'      : avg_60min,
    'delta1'     : delta1,
}

    print('[FITUR]', fitur)
    return fitur


def proses_interval(t_start, t_end, count, durasi):
    print(f'\n=== INTERVAL {t_start}-{t_end} | count={count} durasi={durasi}s ===')

    fitur = hitung_fitur(count, durasi)
    X     = pd.DataFrame([fitur])

    klasifikasi = str(model_klasifikasi.predict(X)[0]).lower()
    print(f'[AI] Klasifikasi : {klasifikasi}')

    prediksi = str(model_prediksi.predict(X)[0]).lower() if model_prediksi else '-'
    print(f'[AI] Prediksi 1j : {prediksi}')

    freq = FREQ_MAP.get(klasifikasi, FREQ_MAP['rendah'])
    print(f'[AI] Frekuensi   : {freq["freq_min"]}-{freq["freq_max"]} Hz')

    riwayat_count.append(count)

    return {
        't_start': t_start, 't_end': t_end,
        'count': count, 'durasi': durasi,
        'klasifikasi': klasifikasi,
        'freq_min': freq['freq_min'], 'freq_max': freq['freq_max'],
        'prediksi': prediksi,
    }


def kirim_ke_sheets(hasil, nomor):
    tanggal   = datetime.now().strftime('%d/%m/%Y')
    frekuensi = f"{hasil['freq_min']} - {hasil['freq_max']} Hz"
    baris = [
        tanggal,
        hasil['t_start'] + ':00',
        hasil['t_end']   + ':00',
        nomor,
        hasil['count'],
        hasil['durasi'],
        hasil['klasifikasi'],
        frekuensi,
        hasil['prediksi'],
    ]
    worksheet.append_row(baris)
    print(f'[SHEETS] Baris {nomor} ditambahkan.')


def kirim_ke_mqtt(hasil):
    payload = json.dumps({
        'klasifikasi'  : hasil['klasifikasi'],
        'freq_min'     : hasil['freq_min'],
        'freq_max'     : hasil['freq_max'],
        'prediksi_1jam': hasil['prediksi'],
    })
    mqtt_client.publish(MQTT_TOPIC, payload)
    print(f'[MQTT] Published: {payload}')


print('Fungsi pemrosesan siap.')

Fungsi pemrosesan siap.


## SEL 8 – Flask server

In [ ]:
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/predict', methods=['POST'])
def predict():
    global nomor_interval
    try:
        data    = request.get_json(force=True)
        t_start = data.get('t_start', '--:--')
        t_end   = data.get('t_end',   '--:--')
        count   = int(data.get('count',  0))
        durasi  = int(data.get('durasi', 0))

        nomor_interval += 1
        hasil = proses_interval(t_start, t_end, count, durasi)
        kirim_ke_sheets(hasil, nomor_interval)
        kirim_ke_mqtt(hasil)

        return jsonify({
            'status'     : 'ok',
            'interval_ke': nomor_interval,
            'klasifikasi': hasil['klasifikasi'],
            'prediksi'   : hasil['prediksi'],
        }), 200
    except Exception as e:
        print(f'[ERROR] {e}')
        return jsonify({'status': 'error', 'message': str(e)}), 500

@app.route('/ping', methods=['GET'])
def ping():
    return jsonify({'status': 'ok', 'message': 'Server aktif'}), 200

def run_flask():
    app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
print('Flask server berjalan di port 5000.')

Flask server berjalan di port 5000.
 * Serving Flask app '__main__'
 * Debug mode: off


## SEL 9 – Buat tunnel ngrok & kirim URL ke Apps Script

URL ngrok dikirim ke Apps Script via HTTP POST.
Apps Script menyimpannya ke sel B2 tab 'Info Server'.
ESP32 #2 membaca URL dari Apps Script saat boot.

In [ ]:
from pyngrok import ngrok, conf
import requests as req
import time

conf.get_default().auth_token = NGROK_AUTHTOKEN

ngrok.kill()
time.sleep(1)

tunnel    = ngrok.connect(5000)
colab_url = tunnel.public_url
full_url  = f'{colab_url}/predict'

print('=' * 60)
print(f'  URL ngrok terbaru:')
print(f'  {full_url}')
print('=' * 60)

# Kirim URL ke Apps Script via HTTP POST
# Apps Script akan menyimpannya ke sel B2 Sheets
print('\nMengirim URL ke Apps Script...')
try:
    resp = req.post(
        APPS_SCRIPT_URL,
        json={'url': full_url},
        timeout=15
    )
    result = resp.json()
    if result.get('status') == 'ok':
        print(f'[APPS SCRIPT] URL berhasil disimpan ke Sheets.')
        print(f'[APPS SCRIPT] ESP32 #2 akan baca URL ini saat boot.')
    else:
        print(f'[APPS SCRIPT] Gagal: {result.get("message")}')
except Exception as e:
    print(f'[APPS SCRIPT] Error: {e}')
    print('Pastikan APPS_SCRIPT_URL sudah diisi dengan benar di SEL 2.')

[MQTT] Terhubung ke broker.emqx.io
[MQTT] Terhubung ke broker.emqx.io
  URL ngrok terbaru:
  https://directory-perkiness-nutty.ngrok-free.dev/predict

Mengirim URL ke Apps Script...
[MQTT] Terhubung ke broker.emqx.io
[APPS SCRIPT] URL berhasil disimpan ke Sheets.
[APPS SCRIPT] ESP32 #2 akan baca URL ini saat boot.


## SEL 10 – (Opsional) Test manual tanpa ESP32

In [ ]:
import requests

test_payload = {
    't_start': '09:00',
    't_end'  : '09:10',
    'count'  : 8,
    'durasi' : 35
}

resp = requests.post(full_url, json=test_payload)
print('Status :', resp.status_code)
print('Respons:', resp.json())


=== INTERVAL 09:00-09:10 | count=8 durasi=35s ===
[FITUR] {'count': 8, 'durasi aktif': 35, 'lag1': 0, 'lag2': 0, 'lag3': 0, 'lag4': 0, 'lag5': 0, 'lag6': 0, 'avg30': np.float64(0.0), 'avg60': np.float64(0.0), 'delta1': 8}
[AI] Klasifikasi : 1
[AI] Prediksi 1j : 5.297211418859682
[AI] Frekuensi   : 20000-20167 Hz


INFO:werkzeug:127.0.0.1 - - [29/May/2026 06:39:35] "POST /predict HTTP/1.1" 200 -


[SHEETS] Baris 1 ditambahkan.
[MQTT] Published: {"klasifikasi": "1", "freq_min": 20000, "freq_max": 20167, "prediksi_1jam": "5.297211418859682"}
Status : 200
Respons: {'interval_ke': 1, 'klasifikasi': '1', 'prediksi': '5.297211418859682', 'status': 'ok'}


---
## Catatan penting

- **Jangan tutup tab Colab** saat sistem berjalan.
- **Setiap restart**: jalankan ulang SEL 1-9. URL ngrok baru otomatis dikirim ke Apps Script → disimpan ke Sheets → ESP32 #2 baca saat boot. Tidak perlu upload ulang kode ESP32.
- **APPS_SCRIPT_URL bersifat permanen** — tidak berubah meskipun Colab restart.
- **Urutan sel wajib**: 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7 -> 8 -> 9.